# Finetune Llama-3 with LLaMA Factory


## Install Dependencies

In [1]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (360/360), done.
remote: Compressing objects: 100% (279/279), done.
remote: Total 360 (delta 79), reused 274 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (360/360), 9.94 MiB | 30.95 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/LLaMA-Factory
assets/       evaluation/  MANIFEST.in     requirements.txt  tests/
CITATION.cff  examples/    pyproject.toml  scripts/
data/         LICENSE      README.md       setup.py
docker/       Makefile     README_zh.md    src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!pip install deepspeed==0.16.9

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.8/422.8 kB 33.9 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.16.9-py3-none-any.whl size=1644431 sha256=6143e040cd44dbdcc67b2fb38632696ebadd99ad738a19341661636f0447b625
  Stored in directory: /root/.cache/pip/wheels/b9/ac/79/a454936d70b601056346a28d4347456cbdf57587851eab3cf3
Successfully built deepspeed


### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
%cd /content

/content


In [3]:
!git clone https://github.com/StibiumT16/Robust-Fine-tuning

Cloning into 'Robust-Fine-tuning'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 53 (delta 21), reused 48 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 524.82 KiB | 5.14 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [ ]:
# !cp -r /content/drive/MyDrive/nlp_project/e5/train /content/Robust-Fine-tuning/data/e5/train
# !cp -r /content/drive/MyDrive/nlp_project/lora_weights /content/lora_weights

In [ ]:
# !tar -xvzf /content/lora_weights/lora_weights.tar.gz

lora_weights/
lora_weights/rbft_qwen/
lora_weights/rbft_qwen/adapter_config.json
lora_weights/rbft_qwen/vocab.json
lora_weights/rbft_qwen/merges.txt
lora_weights/rbft_qwen/adapter_model.safetensors
lora_weights/rbft_qwen/added_tokens.json
lora_weights/rbft_qwen/tokenizer.json
lora_weights/rbft_qwen/special_tokens_map.json
lora_weights/rbft_qwen/tokenizer_config.json
lora_weights/rbft_llama/
lora_weights/rbft_llama/tokenizer.json
lora_weights/rbft_llama/adapter_model.safetensors
lora_weights/rbft_llama/special_tokens_map.json
lora_weights/rbft_llama/adapter_config.json
lora_weights/rbft_llama/tokenizer_config.json


# Need token for Huggingface LLaMA download

In [ ]:
!huggingface-cli login --token 

In [ ]:
# !huggingface-cli download meta-llama/Llama-3.2-3B-Instruct --local-dir /content/meta-llama/Llama-3.2-3B-Instruct

Fetching 16 files:   0% 0/16 [00:00<?, ?it/s]Downloading 'config.json' to '/content/meta-llama/Llama-3.2-3B-Instruct/.cache/huggingface/download/8_PA_wEVGiVa2goH2H4KQOQpvVY=.a5a40fa6da567ab026a5a2bf37125a90182be07d.incomplete'

config.json: 100% 878/878 [00:00<00:00, 5.34MB/s]
Download complete. Moving file to /content/meta-llama/Llama-3.2-3B-Instruct/config.json

model-00002-of-00002.safetensors:   0% 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0% 0.00/4.97G [00:00<?, ?B/s]


USE_POLICY.md: 100% 6.02k/6.02k [00:00<00:00, 33.7MB/s]
Download complete. Moving file to /content/meta-llama/Llama-3.2-3B-Instruct/USE_POLICY.md



LICENSE.txt: 100% 7.71k/7.71k [00:00<00:00, 43.1MB/s]



generation_config.json: 100% 189/189 [00:00<00:00, 2.00MB/s]
Download complete. Moving file to /content/meta-llama/Llama-3.2-3B-Instruct/LICENSE.txt
Download complete. Moving file to /content/meta-llama/Llama-3.2-3B-Instruct/generation_config.json



README.md:   0% 0.00/41.7k [00:00<?, ?B/s

In [5]:
!huggingface-cli download loganchew/rbft_llama3

Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Downloading 'generation_config.json' to '/root/.cache/huggingface/hub/models--loganchew--rbft_llama3/blobs/86546ab87e4eb79c96f00ffe17ad89ca00e2ecc1.incomplete'

config.json: 100% 873/873 [00:00<00:00, 6.61MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--loganchew--rbft_llama3/blobs/8cac94b68fcf02a50f3ba263bd9d3b9919ca3a05

generation_config.json: 100% 184/184 [00:00<00:00, 1.77MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--loganchew--rbft_llama3/blobs/86546ab87e4eb79c96f00ffe17ad89ca00e2ecc1

Modelfile: 100% 498/498 [00:00<00:00, 4.90MB/s]
Download complete. Moving file to /root/.cache/huggingface/hub/models--loganchew--rbft_llama3/blobs/3cb06b0d7d0ff4f61581b8b3f88ef651462a1dae

model-00001-of-00004.safetensors:   0% 0.00/2.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0% 0.00/1.96G [00:00<?, ?B/s]


.gitattributes: 100% 1.57k/1.57k [00:00<00:00, 15.4MB/s]
Download complete

In [6]:
!cp -r /content/drive/MyDrive/nlp_project/e5/train_musique/rbft.json /content/Robust-Fine-tuning/rbft/data/

In [ ]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, PreTrainedModel, PretrainedConfig
from huggingface_hub import HfApi, create_repo, upload_folder
import os

# --- 1. Define a very small custom model and its configuration ---

# First, define a custom configuration for your model
class MyTinyModelConfig(PretrainedConfig):
    model_type = "my_tiny_model"

    def __init__(self, vocab_size=100, hidden_size=16, num_labels=2, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_labels = num_labels

# Now, define your custom model
class MyTinyModel(PreTrainedModel):
    config_class = MyTinyModelConfig
    base_model_prefix = "my_tiny_model"

    def __init__(self, config):
        super().__init__(config)
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, input_ids, labels=None):
        embedded = self.embedding(input_ids)
        # For simplicity, let's just average the embeddings
        pooled = torch.mean(embedded, dim=1)
        logits = self.classifier(pooled)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))
        return (loss, logits) if loss is not None else logits

# --- 2. Create an instance of the model and a dummy tokenizer ---

# Create an instance of the custom configuration
config = MyTinyModelConfig(vocab_size=100, hidden_size=16, num_labels=2)
model = MyTinyModel(config)

# For a small model, you might also want a basic tokenizer.
# We'll use a pre-trained one for simplicity, but you could train your own.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # Just to get a functional tokenizer

# --- 3. Save the model locally ---

# Define a local directory to save your model
save_directory = "./my_tiny_model_local"
os.makedirs(save_directory, exist_ok=True)

# Save the model and tokenizer
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer saved to {save_directory}")
print(os.listdir(save_directory))

# --- 4. Push the model to Hugging Face Hub ---

# # Replace with your Hugging Face username and desired repository name
# your_username = "your_username" # e.g., "john_doe"
# model_name = "my-tiny-example-model"
# repo_id = f"{your_username}/{model_name}"

# api = HfApi()

# # Create a new repository on the Hugging Face Hub
# # Set private=True if you want it to be a private repository
# create_repo(repo_id=repo_id, exist_ok=True, private=False)
# print(f"Repository '{repo_id}' created or already exists on Hugging Face Hub.")

# # Upload the saved files to the repository
# upload_folder(
#     folder_path=save_directory,
#     repo_id=repo_id,
#     repo_type="model",
#     commit_message="Initial upload of MyTinyModel"
# )

# print(f"Model successfully pushed to Hugging Face Hub: https://huggingface.co/{repo_id}")

# # --- 5. (Optional) Load the model from the Hugging Face Hub to verify ---

# print("\n--- Verifying by loading from Hugging Face Hub ---")
# try:
#     # AutoModel will try to infer the model class from the config.json
#     loaded_model = MyTinyModel.from_pretrained(repo_id)
#     loaded_tokenizer = AutoTokenizer.from_pretrained(repo_id)

#     print("Model and tokenizer loaded successfully from Hugging Face Hub!")
#     print(f"Loaded model config: {loaded_model.config}")

#     # Example inference
#     inputs = tokenizer("Hello world!", return_tensors="pt")
#     outputs = loaded_model(input_ids=inputs.input_ids)
#     print(f"Inference output logits: {outputs}")

# except Exception as e:
#     print(f"Error loading model from Hub: {e}")
#     print("Make sure 'MyTinyModel' and 'MyTinyModelConfig' are defined or imported when loading.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Model and tokenizer saved to ./my_tiny_model_local
['model.safetensors', 'config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json', 'tokenizer_config.json']


## Infer the fine-tuned model

In [ ]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /content/LLaMA-Factory/

args = dict(
  model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="llama3_lora",                        # load the saved LoRA adapters
  template="llama3",                                         # same to the one in training
  finetuning_type="lora",                                    # same to the one in training
)
chat_model = ChatModel(args)

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = []
    torch_gc()
    print("History has been removed.")
    continue

  messages.append({"role": "user", "content": query})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()